# Driving the HITL review graph  (Phase 3)

Same outcome as `review.ipynb`, but run as a real **LangGraph** graph
(`graphs/review_pipeline.py`): it runs Model A (extract, as a subgraph) and Model B (review),
then **pauses** at a human `interrupt()`. You examine the diff, then **resume** it with your
decisions — the graph adjudicates your flags (apply-or-rebut) and finishes.

Uses a checkpointer, so the pause/resume state lives across cells. Spec: `specs/review_and_gold.md`.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from clinical import (
    get_transcript,
    DEFAULT_MODEL,
    REVIEWER_MODEL,
    show_review_diff,
    show_symptom_evidence_matrix,
    build_gold_record,
    save_gold,
    gold_progress,
)
from graphs.review_pipeline import build_review_graph

GRAPH = build_review_graph(checkpointer=MemorySaver())  # real models
ENCOUNTER_DATE = "27.11.2025"
print("extractor (A):", DEFAULT_MODEL)
print("reviewer/adjudicator (B):", REVIEWER_MODEL)

## 1. Pick a transcript + a thread id

The `thread_id` keys the checkpoint. To re-run the **same** transcript, restart the kernel or
change the id (resuming a finished thread won't pause again).

In [ ]:
TID = "transcript_1"
transcript = get_transcript(TID)
config = {"configurable": {"thread_id": TID}}
print(transcript)

## 2. Run until the human step  ⚠️ 2 API calls (A extract, B review)

The graph runs `extract → review` and then **pauses** at `human_review`.

In [ ]:
out = GRAPH.invoke({"transcript": transcript, "encounter_date": ENCOUNTER_DATE}, config)
assert "__interrupt__" in out, "expected the graph to pause for you"

snap = GRAPH.get_state(config)
extraction = snap.values["extraction"]
critique = snap.values["critique"]
print(f"paused at {snap.next}. reviewer proposed {len(critique.changes)} change(s): {critique.overall}")

## 3. Examine the diff at the pause — 🟩 A cited · 🟧 B says missed

In [ ]:
show_review_diff(transcript, extraction, critique)

## 4. The reviewer's proposals

In [ ]:
for i, ch in enumerate(critique.changes):
    print(f"[{i}] {ch.kind:13} {ch.problem} · field={ch.field} → {ch.text}")
    if ch.excerpt:
        print(f"      “{ch.excerpt}”")

## 5. Resume the graph with your decisions  ⚠️ 1 API call per flag

`ACCEPT` = reviewer proposals you agree with. `FLAGS` = things **you** think were missed — the
adjudicator will either add each (with a verbatim excerpt) or push back with a rebuttal.

In [ ]:
ACCEPT = []    # e.g. [0, 2]
FLAGS = []     # e.g. ["patient denied fever", "pain radiates to the back"]

final_state = GRAPH.invoke(Command(resume={"accepted": ACCEPT, "flags": FLAGS}), config)
final_extraction = final_state["final_extraction"]
print(f"final problems: {len(final_extraction.problems)}")
for r in final_state.get("rebuttals", []):
    print(f"REBUTTAL — {r['flag']!r}: {r['rebuttal']}")

## 6. The merged result

In [ ]:
show_symptom_evidence_matrix(final_extraction)

## 7. Save as gold

In [ ]:
decisions = [{"index": i, "accepted": (i in ACCEPT)} for i in range(len(critique.changes))]
record = build_gold_record(
    TID, final_extraction, critique, decisions,
    extractor_model=DEFAULT_MODEL, reviewer_model=REVIEWER_MODEL, encounter_date=ENCOUNTER_DATE,
)
print("saved", save_gold(record), "| gold so far:", gold_progress())

## Same graph in Studio

`langgraph dev` from the repo root → open **`review_pipeline`**. Studio surfaces the interrupt and
lets you resume there too — a visual view of the exact flow above.